In [1]:
import pandas as pd
import numpy as np
import os
import mlflow
from sklearn.model_selection import train_test_split
# Load
df = pd.read_csv("housing.csv")

print(df.shape)
print(df.columns)
print(df.describe())

/opt/anaconda3/envs/mlops311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(20640, 10)
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')
          longitude      latitude  housing_median_age   total_rooms  \
count  20640.000000  20640.000000        20640.000000  20640.000000   
mean    -119.569704     35.631861           28.639486   2635.763081   
std        2.003532      2.135952           12.585558   2181.615252   
min     -124.350000     32.540000            1.000000      2.000000   
25%     -121.800000     33.930000           18.000000   1447.750000   
50%     -118.490000     34.260000           29.000000   2127.000000   
75%     -118.010000     37.710000           37.000000   3148.000000   
max     -114.310000     41.950000           52.000000  39320.000000   

       total_bedrooms    population    households  median_income  \
count    20433.000000  20640.000000  20640.000000   20640.000000   
me

In [2]:
# Missing values
print(df.isnull().sum())
# Handle missing values in totalBedrooms by imputing with median
median_bedrooms = df["total_bedrooms"].median()
df["total_bedrooms"] = df["total_bedrooms"].fillna(median_bedrooms)

print(f"Imputed {207} nulls in total_bedrooms with median = {median_bedrooms}")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64
Imputed 207 nulls in total_bedrooms with median = 435.0
Remaining nulls: 0


In [3]:
# One-hot encode to keep it sklearn/FLAML compatible
df = pd.get_dummies(df, columns=["ocean_proximity"], prefix="ocean")
 
# Convert bool columns to int (some FLAML estimators prefer 0/1)
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)
 
print(f"Encoded columns added: {[c for c in df.columns if c.startswith('ocean_')]}")
print(f"Final shape after encoding: {df.shape}")

Encoded columns added: ['ocean_<1H OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']
Final shape after encoding: (20640, 14)


In [4]:
# For reproducibility
SEED = 42
# Split into train, val, test (7-2-1)
# 90 train+val, 10% test
df_trainval, df_test = train_test_split(df, test_size=0.10, random_state=SEED)
 
# From the 90%, carve out 20% of total -> 0.2/0.9 = 0.2222
df_train, df_val = train_test_split(df_trainval, test_size=0.2222, random_state=SEED)
 
print(f"Train : {len(df_train):,} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Val: {len(df_val):,} ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test: {len(df_test):,} ({len(df_test)/len(df)*100:.1f}%)")
 

Train : 14,448 (70.0%)
Val: 4,128 (20.0%)
Test: 2,064 (10.0%)


In [5]:
# Save splits to CSV
os.makedirs(f"splits", exist_ok=True)
df_train.to_csv(f"splits/train.csv", index=False)
df_val.to_csv(f"splits/val.csv", index=False)
df_test.to_csv(f"splits/test.csv", index=False)
 
print(f"Saved to splits/")

Saved to splits/


In [6]:
# Data Corruption Creation from train stats applied to test set (just like scaler or imputer) to simulate production drift
age_std = df_train["housing_median_age"].std()
rooms_std = df_train["total_rooms"].std()
bedrms_std = df_train["total_bedrooms"].std()
 
print(f"Corruption scales (from train dataframe):")
print(f"housin_median_age noise scale : {1.5 * age_std:.4f}")
print(f"total_rooms shift : +{1.5 * rooms_std:.4f}")
print(f"total_bedrooms shift : +{1.5 * bedrms_std:.4f}")
 
def corrupt_batch(df_source, seed):
    df_c = df_source.copy()
    rng  = np.random.default_rng(seed)
    df_c["housing_median_age"] += rng.normal(loc=0, scale=1.5 * age_std,    size=len(df_c))
    df_c["total_rooms"] += 1.5 * rooms_std
    df_c["total_bedrooms"] += 1.5 * bedrms_std
    return df_c
 
os.makedirs(f"production_batches", exist_ok=True)
# Each batch has a different seed to give each batch a unique pattern of corruption to avoid carbon copies
# Batch 1 - clean (no drift, baseline check)
df_batch1 = df_test.copy()
df_batch1.to_csv(f"production_batches/batch1_clean.csv", index=False)
 
# Batch 2 - fully corrupted (strong drift, Evidently should flag this)
df_batch2 = corrupt_batch(df_test, seed=1)
df_batch2.to_csv(f"production_batches/batch2_corrupted.csv", index=False)
 
# Batch 3 - 50/50 mix (partial/realistic drift)
half = len(df_test) // 2
df_batch3 = pd.concat([
    df_test.iloc[:half].copy(),
    corrupt_batch(df_test.iloc[half:], seed=2)
], ignore_index=True)
df_batch3.to_csv(f"production_batches/batch3_mixed.csv", index=False)
 
print("Saved 3 production batches.")

Corruption scales (from train dataframe):
housin_median_age noise scale : 18.8990
total_rooms shift : +3298.4842
total_bedrooms shift : +633.2465
Saved 3 production batches.


In [7]:
# Sanity check the batches by comparing stats
def print_batch_stats(df, name):
    print(f"Stats for {name}:")
    print(df[["housing_median_age", "total_rooms", "total_bedrooms"]].describe())
    print()
print_batch_stats(df_test, "Original Test Set")
print_batch_stats(df_batch1, "Batch 1 - Clean")
print_batch_stats(df_batch2, "Batch 2 - Corrupted")
print_batch_stats(df_batch3, "Batch 3 - Mixed")

Stats for Original Test Set:
       housing_median_age   total_rooms  total_bedrooms
count         2064.000000   2064.000000     2064.000000
mean            28.891473   2618.275678      527.619671
std             12.486613   2179.862623      421.486557
min              1.000000      6.000000        2.000000
25%             19.000000   1429.750000      309.000000
50%             29.000000   2116.000000      435.000000
75%             37.000000   3112.500000      609.250000
max             52.000000  24121.000000     5419.000000

Stats for Batch 1 - Clean:
       housing_median_age   total_rooms  total_bedrooms
count         2064.000000   2064.000000     2064.000000
mean            28.891473   2618.275678      527.619671
std             12.486613   2179.862623      421.486557
min              1.000000      6.000000        2.000000
25%             19.000000   1429.750000      309.000000
50%             29.000000   2116.000000      435.000000
75%             37.000000   3112.500000      60

In [8]:
# Log the batches and experiments as MLflow artifacts for easy access in the next steps
mlflow.set_experiment("california_housing_automl")
 
with mlflow.start_run(run_name="phase1_data_prep"):
 
    # Split params
    mlflow.log_params({
        "random_seed": SEED,
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "split_ratio": "70-20-10",
        "total_features": df_train.shape[1] - 1,
        "totalBedrooms_nulls_imputed": 207,
        "totalBedrooms_impute_value": round(median_bedrooms, 4),
    })
 
    # Corruption params (reproducibility)
    mlflow.log_params({
        "corrupt_age_noise_scale": round(1.5 * age_std, 4),
        "corrupt_rooms_shift": round(1.5 * rooms_std, 4),
        "corrupt_bedrms_shift": round(1.5 * bedrms_std, 4),
    })
 
    # Splits
    mlflow.log_artifact(f"splits/train.csv", artifact_path="splits")
    mlflow.log_artifact(f"splits/val.csv", artifact_path="splits")
    mlflow.log_artifact(f"splits/test.csv", artifact_path="splits")
 
    # Production batches
    mlflow.log_artifact(f"production_batches/batch1_clean.csv", artifact_path="batches")
    mlflow.log_artifact(f"production_batches/batch2_corrupted.csv", artifact_path="batches")
    mlflow.log_artifact(f"production_batches/batch3_mixed.csv", artifact_path="batches")
 
    print(f"MLflow run logged under experiment: california_housing_automl")

MLflow run logged under experiment: california_housing_automl


In [9]:
import mlflow.sklearn
import time
from flaml import AutoML
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
# AutoML settings
TARGET = "median_house_value"
TIME_BUDGET = 300 # seconds
SEED = 42
ESTIMATORS = ["xgboost", "lgbm", "rf", "extra_tree", "catboost"]
EXPERIMENT  = "california_housing_automl"
# Load the train and val splits
X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]
 
X_val = df_val.drop(columns=[TARGET])
y_val = df_val[TARGET]
 
print(f"Train: {X_train.shape} | Val: {X_val.shape}")
 
# Clean column names - XGBoost rejects [, ], < characters
def clean_cols(df):
    df.columns = (
        df.columns
        .str.replace("[", "_", regex=False)
        .str.replace("]", "", regex=False)
        .str.replace("<", "lt", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df
 
X_train = clean_cols(X_train)
X_val = clean_cols(X_val)
 
print(f"Features: {X_train.columns.tolist()}")

Train: (14448, 13) | Val: (4128, 13)
Features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_lt1H_OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR_BAY', 'ocean_NEAR_OCEAN']


In [11]:
# Run FLAML AutoML
automl = AutoML()
 
flaml_settings = {
    "task": "regression",
    "metric": "rmse",
    "time_budget": TIME_BUDGET,
    "estimator_list": ESTIMATORS,
    "seed": SEED,
    "eval_method": "holdout", # use our val dataset explicitly
    "split_ratio": 0.2, # ignored when X_val is passed
    "log_file_name": "flaml_run.log", # saves trial history to disk
    "verbose": 2, # show trial progress, 2 is more detailed than 1
}
 
start = time.time()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    **flaml_settings
)
elapsed = time.time() - start
 
print(f"\nFLAML finished in {elapsed:.1f}s")


FLAML finished in 300.1s


In [12]:
# Evaluate the best model on validation set
best_model = automl.model.estimator
val_preds = automl.predict(X_val)
 
val_rmse = root_mean_squared_error(y_val, val_preds)
val_mae = mean_absolute_error(y_val, val_preds)
val_r2 = r2_score(y_val, val_preds)
 
print(f"Best estimator: {automl.best_estimator}")
print(f"Val RMSE: {val_rmse:,.2f}")
print(f"Val MAE: {val_mae:,.2f}")
print(f"Val R^2: {val_r2:.4f}")
print(f"\nBest hyperparameters:\n{automl.best_config}")

Best estimator: xgboost
Val RMSE: 45,790.01
Val MAE: 29,405.29
Val R^2: 0.8354

Best hyperparameters:
{'n_estimators': 4235, 'max_leaves': 9, 'min_child_weight': np.float64(0.44289062123590356), 'learning_rate': np.float64(0.07778407877277206), 'subsample': np.float64(0.9437538377225084), 'colsample_bylevel': np.float64(0.9647949395335206), 'colsample_bytree': np.float64(0.9877878456495247), 'reg_alpha': np.float64(0.008485088595278224), 'reg_lambda': np.float64(0.7439176476353904)}


In [14]:
# Build leaderboard from FLAML trials
import json
leaderboard_rows = [] # list of dicts to create dataframe, each dict is a trial record
if os.path.exists("flaml_run.log"):
    with open("flaml_run.log", "r") as f:
        for line in f:
            try:
                record = json.loads(line.strip())
                leaderboard_rows.append({
                    "estimator": record.get("learner", "unknown"),
                    "val_rmse": record.get("validation_loss", None),  # correct field name
                    "train_time": round(record.get("trial_time", 0), 3), # correct field name
                    "config": str(record.get("config", {})),
                })
            except Exception:
                continue

df_leaderboard = pd.DataFrame(leaderboard_rows)

if not df_leaderboard.empty:
    df_leaderboard = (
        df_leaderboard
        .dropna(subset=["val_rmse"])
        .sort_values("val_rmse")
        .reset_index(drop=True)
    )
    df_leaderboard.index += 1  # rank starts at 1

    print("\nTop 10 trials:")
    print(df_leaderboard[["estimator", "val_rmse", "train_time"]].head(10).to_string())

    print("\nBest result per estimator type:")
    best_per_type = (
        df_leaderboard
        .groupby("estimator")["val_rmse"]
        .min()
        .sort_values()
        .reset_index()
    )
    best_per_type.columns = ["estimator", "best_val_rmse"]
    print(best_per_type.to_string(index=False))

    df_leaderboard.to_csv("flaml_leaderboard.csv", index=True, index_label="rank")
    print("\nLeaderboard saved to flaml_leaderboard.csv")
else:
    print("Note: Log file not found or empty.")
    print(f"Best estimator confirmed: {automl.best_estimator}")


Top 10 trials:
   estimator      val_rmse  train_time
1    xgboost  45790.013613       6.693
2    xgboost  46072.394141       3.614
3    xgboost  46538.701790       1.351
4   catboost  47035.185286       1.759
5    xgboost  49505.047923       0.517
6    xgboost  50737.084232       0.164
7    xgboost  53647.349532       0.081
8    xgboost  61196.901327       0.041
9       lgbm  84748.761463       0.016
10      lgbm  95255.675903       0.024

Best result per estimator type:
estimator  best_val_rmse
  xgboost   45790.013613
 catboost   47035.185286
     lgbm   84748.761463

Leaderboard saved to flaml_leaderboard.csv


In [15]:
# Log findings to MLFlow
import joblib

mlflow.set_experiment(EXPERIMENT)
 
with mlflow.start_run(run_name="phase2_flaml_automl") as parent_run:
 
    # Parent run: summary of the AutoML search and best model
    mlflow.log_params({
        "time_budget_sec": TIME_BUDGET,
        "estimators": str(ESTIMATORS),
        "metric": "rmse",
        "seed": SEED,
        "actual_time_sec": round(elapsed, 1),
    })
    mlflow.log_metrics({
        "best_val_rmse": val_rmse,
        "best_val_mae": val_mae,
        "best_val_r2": val_r2,
    })
    mlflow.log_param("best_estimator", automl.best_estimator)
    mlflow.log_param("best_config", str(automl.best_config))
 
    # Save & log the best model
    os.makedirs("models", exist_ok=True)
    joblib.dump(automl, "models/flaml_automl_best.pkl")
    mlflow.log_artifact("models/flaml_automl_best.pkl", artifact_path="models")
 
    # Log leaderboard if it was built
    if os.path.exists("data/flaml_leaderboard.csv"):
        mlflow.log_artifact("data/flaml_leaderboard.csv", artifact_path="leaderboard")
 
    # Log FLAML trial log
    if os.path.exists("flaml_run.log"):
        mlflow.log_artifact("flaml_run.log", artifact_path="logs")
 
    # Child runs: one per estimator type (best config per type)
    if not df_leaderboard.empty:
        for _, row in best_per_type.iterrows():
            with mlflow.start_run(
                run_name=f"flaml_candidate_{row['estimator']}",
                nested=True
            ):
                mlflow.log_param("estimator", row["estimator"])
                mlflow.log_metric("best_val_rmse", row["best_val_rmse"])
                mlflow.log_param("source", "flaml_automl_search")
 
    parent_run_id = parent_run.info.run_id
    print(f"Parent MLflow run ID: {parent_run_id}")
    print(f"Experiment: {EXPERIMENT}")

Parent MLflow run ID: edebda8d5f724c6d9d72ff5de61b3afe
Experiment: california_housing_automl


In [16]:
# Best Model and Hyperparameters from the FLAML AutoML search
print(automl.best_config)
print(automl.best_estimator)

{'n_estimators': 4235, 'max_leaves': 9, 'min_child_weight': np.float64(0.44289062123590356), 'learning_rate': np.float64(0.07778407877277206), 'subsample': np.float64(0.9437538377225084), 'colsample_bylevel': np.float64(0.9647949395335206), 'colsample_bytree': np.float64(0.9877878456495247), 'reg_alpha': np.float64(0.008485088595278224), 'reg_lambda': np.float64(0.7439176476353904)}
xgboost


In [18]:
# Refining top models with more time and focused search
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
N_TRIALS = 20 # Optuna trials per model
# Optuna Objective Functions
# Each starts from FLAML's best config as the center of the search space (warm start)
# XGBoost objective
def objective_xgboost(trial):
    params = {
        # FLAML best was n_estimators=4235 — search around it
        "n_estimators": trial.suggest_int("n_estimators", 2000, 6000),
        "max_leaves": trial.suggest_int("max_leaves", 6, 64),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 10.0, log=True),
        # FLAML best lr=0.0778 — search tighter range around it
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 2.0, log=True),
        "random_state": SEED, "n_jobs": -1, "verbosity": 0,
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    return root_mean_squared_error(y_val, model.predict(X_val))
# CatBoost objective
def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":trial.suggest_float("random_strength", 0.5, 2.0),
        "random_seed": SEED, "verbose": 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    return root_mean_squared_error(y_val, model.predict(X_val))
# LightGBM objective
def objective_lgbm(trial):
    # LightGBM got no fair search in FLAML - full range here
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 5000),
        "num_leaves":  trial.suggest_int("num_leaves", 20, 300),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 2.0, log=True),
        "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    return root_mean_squared_error(y_val, model.predict(X_val))

In [19]:
# Run Optuna studies for each model type
objectives = {
    "xgboost":   objective_xgboost,
    "catboost":  objective_catboost,
    "lgbm":      objective_lgbm,
}

best_results = {}   # name -> {rmse, mae, r2, params, model}
 
mlflow.set_experiment(EXPERIMENT)
 
with mlflow.start_run(run_name="phase3_refinement") as parent_run:
 
    for name, objective in objectives.items():
        print(f"\n {name.upper()} ({N_TRIALS} trials)")
 
        study = optuna.create_study(
            direction="minimize",
            sampler=TPESampler(seed=SEED),
            study_name=f"{name}_refinement"
        )
 
        # Warm-start XGBoost with FLAML's best config
        if name == "xgboost":
            study.enqueue_trial({
                "n_estimators": 4235,
                "max_leaves": 9,
                "min_child_weight": 0.44289062123590356,
                "learning_rate": 0.07778407877277206,
                "subsample": 0.9437538377225084,
                "colsample_bylevel": 0.9647949395335206,
                "colsample_bytree": 0.9877878456495247,
                "reg_alpha": 0.008485088595278224,
                "reg_lambda": 0.7439176476353904,
            })
 
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
 
        best_params = study.best_params
        best_rmse = study.best_value
 
        # Refit final model with best params on full train set
        print(f"Best RMSE: {best_rmse:,.2f}")
        print(f"Best params: {best_params}")
 
        if name == "xgboost":
            final_model = XGBRegressor(**best_params, random_state=SEED, n_jobs=-1, verbosity=0)
        elif name == "catboost":
            final_model = CatBoostRegressor(**best_params, random_seed=SEED, verbose=0)
        elif name == "lgbm":
            final_model = LGBMRegressor(**best_params, random_state=SEED, n_jobs=-1, verbose=-1)
 
        final_model.fit(X_train, y_train)
        val_preds = final_model.predict(X_val)
 
        rmse = root_mean_squared_error(y_val, val_preds)
        mae = mean_absolute_error(y_val, val_preds)
        r2 = r2_score(y_val, val_preds)
 
        best_results[name] = {
            "rmse": rmse, "mae": mae, "r2": r2,
            "params": best_params, "model": final_model
        }
 
        # Save model
        model_path = f"models/{name}_refined.pkl"
        joblib.dump(final_model, model_path)
 
        # MLflow child run per estimator
        with mlflow.start_run(run_name=f"refined_{name}", nested=True):
            mlflow.log_params({f"opt_{k}": v for k, v in best_params.items()})
            mlflow.log_metrics({"val_rmse": rmse, "val_mae": mae, "val_r2": r2})
            mlflow.log_artifact(model_path, artifact_path="models")
            mlflow.sklearn.log_model(final_model, artifact_path="model_object")

[I 2026-05-20 21:36:45,598] A new study created in memory with name: xgboost_refinement



 XGBOOST (20 trials)


[I 2026-05-20 21:36:50,366] Trial 0 finished with value: 47045.675115899845 and parameters: {'n_estimators': 4235, 'max_leaves': 9, 'min_child_weight': 0.44289062123590356, 'learning_rate': 0.07778407877277206, 'subsample': 0.9437538377225084, 'colsample_bylevel': 0.9647949395335206, 'colsample_bytree': 0.9877878456495247, 'reg_alpha': 0.008485088595278224, 'reg_lambda': 0.7439176476353904}. Best is trial 0 with value: 47045.675115899845.
[I 2026-05-20 21:36:55,955] Trial 1 finished with value: 46660.827702645955 and parameters: {'n_estimators': 3498, 'max_leaves': 62, 'min_child_weight': 2.9106359131330706, 'learning_rate': 0.050591436432963696, 'subsample': 0.7468055921327309, 'colsample_bylevel': 0.7467983561008608, 'colsample_bytree': 0.7174250836504598, 'reg_alpha': 0.29154431891537524, 'reg_lambda': 0.038495830758681175}. Best is trial 1 with value: 46660.827702645955.
[I 2026-05-20 21:36:59,785] Trial 2 finished with value: 47988.19042369151 and parameters: {'n_estimators': 4832

Best RMSE : 45,572.29
Best params: {'n_estimators': 5985, 'max_leaves': 49, 'min_child_weight': 0.3393571235240794, 'learning_rate': 0.01016194200659135, 'subsample': 0.8619396057955684, 'colsample_bylevel': 0.9141849736013782, 'colsample_bytree': 0.888685517254986, 'reg_alpha': 0.0005303410534244374, 'reg_lambda': 0.004756934492669602}


2026/05/20 21:39:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-05-20 21:39:22,534] A new study created in memory with name: catboost_refinement


...

2026/05/20 21:41:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-05-20 21:41:12,967] A new study created in memory with name: lgbm_refinement



 LGBM (20 trials)


[I 2026-05-20 21:41:44,251] Trial 0 finished with value: 48031.88034047881 and parameters: {'n_estimators': 2185, 'num_leaves': 287, 'min_child_samples': 76, 'learning_rate': 0.050591436432963696, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.0001707396743152812, 'reg_lambda': 0.5314343038625025}. Best is trial 0 with value: 48031.88034047881.
[I 2026-05-20 21:42:58,347] Trial 1 finished with value: 47424.12819056 and parameters: {'n_estimators': 3205, 'num_leaves': 218, 'min_child_samples': 11, 'learning_rate': 0.13826189316223855, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.000533703276260396, 'reg_lambda': 0.000614933705708711}. Best is trial 1 with value: 47424.12819056.
[I 2026-05-20 21:43:30,430] Trial 2 finished with value: 46267.6598577701 and parameters: {'n_estimators': 1869, 'num_leaves': 167, 'min_child_samples': 49, 'learning_rate': 0.022004527434741072, 'subsample': 0.8447411578889518, 'c

Best RMSE : 45,691.20
Best params: {'n_estimators': 898, 'num_leaves': 75, 'min_child_samples': 14, 'learning_rate': 0.02413338039141456, 'subsample': 0.7554709158757927, 'colsample_bytree': 0.7085396127095583, 'reg_alpha': 0.20651425578959245, 'reg_lambda': 0.0034229988967457927}


2026/05/20 21:50:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [24]:
# Compare and Select Champion Model
rows = [{"estimator": k, "val_rmse": v["rmse"], "val_mae": v["mae"], "val_r2": v["r2"]}
            for k, v in best_results.items()]
df_results = pd.DataFrame(rows).sort_values("val_rmse").reset_index(drop=True)
df_results.index += 1
print(df_results.to_string())
 
champion_name = df_results.iloc[0]["estimator"]
champion = best_results[champion_name]
champion_model = champion["model"]
 
print(f"\nChampion: {champion_name.upper()}")
print(f"Val RMSE: {champion['rmse']:,.2f}")
print(f"Val R^2: {champion['r2']:.4f}")

client = mlflow.tracking.MlflowClient()

# Load the already-saved champion from disk
champion_model = joblib.load("models/catboost_refined.pkl")

mlflow.set_experiment("california_housing_automl")

with mlflow.start_run(run_name="phase3_champion_registration") as run:
    
    mlflow.log_params({
        "champion_estimator": "catboost",
        "champion_val_rmse": 44947.37,
        "champion_val_r2": 0.8414,
    })
    
    # Save to disk
    joblib.dump(champion_model, "models/champion_model.pkl")

    # Log and register in the SAME run — no mismatch possible
    mlflow.sklearn.log_model(champion_model, artifact_path="champion")
    
    registered = mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/champion",
        name="california_housing_champion"
    )
    
    client.transition_model_version_stage(
        name="california_housing_champion",
        version=registered.version,
        stage="Production",
        archive_existing_versions=True
    )

    print(f"Run ID : {run.info.run_id}")
    print(f"Registered: california_housing_champion v{registered.version} -> Production")

2026/05/20 22:14:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  estimator      val_rmse       val_mae    val_r2
1  catboost  44947.365505  28811.503209  0.841414
2   xgboost  45572.286719  29316.128867  0.836973
3      lgbm  45691.198518  29208.508994  0.836121

Champion: CATBOOST
Val RMSE: 44,947.37
Val R^2: 0.8414


Registered model 'california_housing_champion' already exists. Creating a new version of this model...
2026/05/20 22:14:56 WARNING mlflow.tracking._model_registry.fluent: Run with id e92e0028bbcc4b8f94599d5d7fa57ebd has no artifacts at artifact path 'champion', registering model based on models:/m-016f384cf5b04572bc51c8bdcf42a5de instead


Run ID : e92e0028bbcc4b8f94599d5d7fa57ebd
Registered: california_housing_champion v1 -> Production


Created version '1' of model 'california_housing_champion'.


In [26]:
# Re-save all splits and batches with clean column names
paths = [
    "splits/train.csv",
    "splits/val.csv",
    "splits/test.csv",
    "production_batches/batch1_clean.csv",
    "production_batches/batch2_corrupted.csv",
    "production_batches/batch3_mixed.csv",
]

for path in paths:
    df = clean_cols(pd.read_csv(path))
    df.to_csv(path, index=False)
    print(f"Sanitized: {path}")

Sanitized: splits/train.csv
Sanitized: splits/val.csv
Sanitized: splits/test.csv
Sanitized: production_batches/batch1_clean.csv
Sanitized: production_batches/batch2_corrupted.csv
Sanitized: production_batches/batch3_mixed.csv


In [38]:
# Final Test Set Evaluation with the Champion Model
df_test  = clean_cols(df_test)
 
X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]
champion = joblib.load("models/champion_model.pkl")
print(f"Champion model loaded: {type(champion).__name__}")
print(f"Test set size: {X_test.shape}")

test_preds = champion.predict(X_test)
 
test_rmse = root_mean_squared_error(y_test, test_preds)
test_mae = mean_absolute_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)
 
print(f"Test RMSE: {test_rmse:,.2f}")
print(f"Test MAE: {test_mae:,.2f}")
print(f"Test R^2: {test_r2:.4f}")

# Reference = training features + target + model predictions on train
# Evidently needs predictions in the reference too for regression reports
train_preds = champion.predict(df_train.drop(columns=[TARGET]))
 
df_reference = df_train.copy()
df_reference["prediction"] = train_preds
 
df_reference.to_csv("reference_data.csv", index=False)
 
print(f"Reference dataset saved: reference_data.csv")
print(f"Shape : {df_reference.shape}")
print(f"Columns: {df_reference.columns.tolist()}")

# Evidently needs predictions in the production batches too — add them now using the champion
batch_files = {
    "batch1_clean": "production_batches/batch1_clean.csv",
    "batch2_corrupted": "production_batches/batch2_corrupted.csv",
    "batch3_mixed": "production_batches/batch3_mixed.csv",
}
 
for batch_name, batch_path in batch_files.items():
    df_batch = clean_cols(pd.read_csv(batch_path))
    X_batch  = df_batch.drop(columns=[TARGET])
 
    df_batch["prediction"] = champion.predict(X_batch)
 
    # Overwrite with predictions included
    df_batch.to_csv(batch_path, index=False)
 
    batch_rmse = root_mean_squared_error(df_batch[TARGET], df_batch["prediction"])
    print(f"{batch_name:<20} RMSE: {batch_rmse:,.2f}  rows: {len(df_batch):,}")

# Log everything to MLflow for easy access in the next steps
mlflow.end_run()  # safety — close any lingering run
mlflow.set_experiment(EXPERIMENT)
 
with mlflow.start_run(run_name="phase4_final_evaluation") as run:
 
    # Final test metrics
    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
    })
 
    mlflow.log_params({
        "champion_estimator": type(champion).__name__,
        "test_set_size": len(df_test),
        "reference_set_size": len(df_reference),
    })
 
    # Log reference dataset — needed later if you re-run monitoring
    mlflow.log_artifact("reference_data.csv", artifact_path="reference")
 
    print(f"MLflow run ID: {run.info.run_id}")
    print(f"Experiment: {EXPERIMENT}")

Champion model loaded: CatBoostRegressor
Test set size: (2064, 13)
Test RMSE: 45,406.67
Test MAE: 29,703.22
Test R^2: 0.8454
Reference dataset saved: reference_data.csv
Shape : (14448, 15)
Columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_lt1H_OCEAN', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR_BAY', 'ocean_NEAR_OCEAN', 'prediction']
batch1_clean         RMSE: 45,406.67  rows: 2,064
batch2_corrupted     RMSE: 74,690.97  rows: 2,064
batch3_mixed         RMSE: 61,164.82  rows: 2,064
MLflow run ID: 22e8caecb93e4b5e8fbb51945794e900
Experiment: california_housing_automl
